## Ở file này ta sẽ lấy dữ liệu từ `https://mobilecity.vn/`

### Import các thư viện cần thiết


In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.webdriver import WebDriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
from typing import Any
from concurrent.futures import ThreadPoolExecutor, wait, as_completed
from threading import Lock
import csv
import os
import json
import random
from tqdm import tqdm

## Đối với trang web mobilecity.vn/dien-thoai


In [8]:
url = "https://mobilecity.vn/dien-thoai"

def get_chrome_options():
    chrome_options = Options()

    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-logging")
    chrome_options.add_argument("--disable-notifications")
    chrome_options.add_argument("--disable-web-security")
    prefs = {"profile.managed_default_content_settings.images": 2}
    chrome_options.add_experimental_option("prefs", prefs)
    chrome_options.page_load_strategy = "eager"
    return chrome_options

driver: WebDriver = webdriver.Chrome(
    service=Service(executable_path=ChromeDriverManager().install()),
    options=get_chrome_options(),
)
driver.get(url)

Hiển thị toàn bộ sản phẩm trước khi lấy link


In [6]:
button_id = "product_view_more"
def click_show_more_btn(button_id: str, max_attempts=20):
    attempts = 0
    while attempts < max_attempts:
        try:
            # Tìm button
            buttons = driver.find_elements(By.ID, button_id)
            if not buttons:
                print("No more 'Show More' buttons found")
                break

            # Lướt đến button
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", buttons[0])
            time.sleep(1)

            driver.execute_script("arguments[0].click();", buttons[0])

            print(f"Clicked button: {attempts+1} times")
            attempts += 1

            # Đợi nội dung load xong
            time.sleep(2)
        except Exception as e:
            print(f"Error: {e}")
            break

driver_wait = WebDriverWait(driver, 10)
button = driver_wait.until(
    EC.element_to_be_clickable((By.ID, button_id))
)
click_show_more_btn(button_id, max_attempts=25)

Clicked button: 1 times
Clicked button: 2 times
Clicked button: 3 times
Clicked button: 4 times
Clicked button: 5 times
Clicked button: 6 times
Clicked button: 7 times
Clicked button: 8 times
Clicked button: 9 times
Clicked button: 10 times
Clicked button: 11 times
Clicked button: 12 times
Clicked button: 13 times
Clicked button: 14 times
Clicked button: 15 times
Clicked button: 16 times
Clicked button: 17 times
Clicked button: 18 times
Clicked button: 19 times
Clicked button: 20 times
Clicked button: 21 times
Clicked button: 22 times
Clicked button: 23 times
Clicked button: 24 times
Clicked button: 25 times


In [8]:
product_elements = driver.find_elements(By.CSS_SELECTOR, ".product-item-left")
print(f"Tổng số sản phẩm: {len(product_elements)}")

Tổng số sản phẩm: 520


In [10]:
print(product_elements[0].text)

Apple Watch Series 5 40mm Viền Nhôm cũ
3.950.000 ₫


### Trích xuất các link sản phẩm từ trang web mobilecity.vn/dien-thoai dựa vào html của các thẻ sản phẩm


In [14]:
product_links = []
link_lock = Lock()

def extract_product_link(product):
    try:
        # Tìm link sản phẩm
        link_element = product.find_element(
            By.CSS_SELECTOR, ".product-item-left p.name a"
        )
        link = link_element.get_attribute("href")

        with link_lock:
            product_links.append(link)

        return link
    except Exception as e:
        print(f"Error: {e} in product {product.text} - skipping")
        return None

num_threads = 12

with ThreadPoolExecutor(max_workers=num_threads) as executor:
    futures = [executor.submit(extract_product_link, product) for product in product_elements]
    
    wait(futures)

print(f"Found {len(product_links)} product links")
print(product_links[:5])

Found 520 product links
['https://mobilecity.vn/dien-thoai/apple-watch-series-5-40mm-vien-nhom-cu.html', 'https://mobilecity.vn/dien-thoai/apple-watch-series-5-44mm-vien-nhom-cu.html', 'https://mobilecity.vn/dien-thoai/samsung-galaxy-a17-5g-chinh-hang.html', 'https://mobilecity.vn/dien-thoai/samsung-galaxy-s21-ultra-cu-camera-108mp.html', 'https://mobilecity.vn/dien-thoai/oneplus-ace-5-cu-99.html']


### Sau khi lấy được các Link hợp lệ, ta tiến hành lưu vào file để đảm bảo không bị mất dữ liệu trong quá trình cào dữ liệu.


In [15]:
mobilecity_links = "../link/product_links_mobilecity.csv"

os.makedirs(os.path.dirname(mobilecity_links), exist_ok=True)

with open(mobilecity_links, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Product Link"])
    writer.writerows([[link] for link in product_links])

### Ta tiến hành cào dữ liệu thô, dữ liệu sẽ lưu theo cấu trúc JSON


In [10]:
def extract_product_common_info(driver: WebDriver, product_link: str) -> dict[str, Any]:
    product_data = {"url": product_link, "name": "", "price": "", "specifications": []}

    print(f"Extracting data from: {product_link}")

    # Lấy tên sản phẩm
    try:
        name_element = driver.find_element(By.CSS_SELECTOR, "p.product-summary-title")
        name = name_element.text.strip()
        if name:
            product_data["name"] = name
            print(f"Found product name: {name}")
    except Exception as e:
        print(f"Error getting product name: {e}")

    time.sleep(1)

    # Lấy giá sản phẩm
    try:
        price_element = driver.find_element(By.CSS_SELECTOR, "p.product-summary-price")
        price = price_element.text.strip()
        if price:
            product_data["price"] = price
            print(f"Found product price: {price}")
    except Exception as e:
        print(f"Error getting product price: {e}")

    # Trích xuất thông số kỹ thuật
    specifications = []

    try:
        driver.execute_script("document.querySelector('.show-lightbox-btn')?.click();")
        WebDriverWait(driver, 5).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, ".product-lightbox-content table")
            )
        )
        time.sleep(1)

        rows = driver.find_elements(By.CSS_SELECTOR, ".product-lightbox-content table tbody tr")

        current_category = "Thông tin chung" # Tên mặc định nếu lỡ hàng đầu tiên không phải là category
        current_details = {}

        for row in rows:
            tds = row.find_elements(By.TAG_NAME, "td")

            # TRƯỜNG HỢP 1: Đây là hàng tiêu đề Danh mục (Chỉ có 1 td, có colspan="2")
            if len(tds) == 1:
                # Lưu category trước đó vào mảng (nếu đã có dữ liệu)
                if current_details:
                    specifications.append({
                        "category": current_category,
                        "details": current_details
                    })
                    # Reset dictionary để chuẩn bị hứng dữ liệu cho category mới
                    current_details = {}

                # Cập nhật tên category mới
                current_category = tds[0].text.strip()

            # TRƯỜNG HỢP 2: Đây là hàng chứa Thông số (Có 2 td)
            elif len(tds) >= 2:
                # Lấy key và xóa luôn dấu hai chấm ":" ở cuối nếu có (VD: "Hệ điều hành:" -> "Hệ điều hành")
                key = tds[0].text.strip().rstrip(':')
                raw_value = tds[1].text.strip()

                # Xử lý giá trị: Nếu có nhiều dòng (do thẻ <br>), tách thành List
                if "\n" in raw_value:
                    value_list =[
                        item.strip() 
                        for item in raw_value.split("\n") 
                        if item.strip()
                    ]
                    current_details[key] = value_list
                else:
                    current_details[key] = raw_value

        if current_details:
            specifications.append({
                "category": current_category,
                "details": current_details
            })

    except Exception as e:
        print(f"Error during specification extraction: {e}")

    driver.execute_script("document.querySelector('.close-lightbox-btn')?.click();")

    if specifications:
        product_data["specifications"] = specifications

    print(f"product name common data: {product_data}")
    return product_data

### Lấy dữ liệu và lưu dữ liệu vào file json


In [5]:
json_product_detail_folder = "../data/json"
def getAndSaveProductInfo(driver: WebDriver, product_link: str) -> dict[str, Any]:
    """
    Lấy thông tin sản phẩm từ trang product_link bằng cách cào thông tin common và variant, sau đó merge lại thành 1 dict.
    
    Returns:
    - Dictionary chứa thông tin sản phẩm, bao gồm:
         - url, specifications, variant_prices
    """
    driver.get(product_link)
    time.sleep(2)
    
    product_common_data = extract_product_common_info(driver, product_link)
    if product_common_data is None:
        product_common_data = {"url": product_link, "specifications": {}}
    
    file_name = product_link.split("/")[-1].replace(".html", "")
    
    save_path = os.path.join(json_product_detail_folder, file_name + ".json")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(product_common_data, f, indent=4, ensure_ascii=False)
        
    return product_common_data

Test hàm với 1 link sản phẩm ngẫu nhiên triong file product_links.csv


In [18]:
getAndSaveProductInfo(
    driver, "https://mobilecity.vn/dien-thoai/samsung-galaxy-a17-5g-chinh-hang.html"
)

Extracting data from: https://mobilecity.vn/dien-thoai/samsung-galaxy-a17-5g-chinh-hang.html
Found product name: Samsung Galaxy A17 5G Chính hãng
Found product price: 4.750.000 ₫
product name common data: {'url': 'https://mobilecity.vn/dien-thoai/samsung-galaxy-a17-5g-chinh-hang.html', 'name': 'Samsung Galaxy A17 5G Chính hãng', 'price': '4.750.000 ₫', 'specifications': [{'category': 'Thông tin chung', 'details': {'Hệ điều hành': ['Android 15, One UI 7', 'Hỗ trợ cập nhật 6 bản Android chính'], 'Ngôn ngữ': 'Tiếng Việt, Đa ngôn ngữ'}}, {'category': 'Màn hình', 'details': {'Loại màn hình': 'Super AMOLED', 'Màu màn hình': '16 triệu màu', 'Chuẩn màn hình': ['Super AMOLED, 90Hz, 800 nits (HBM)', '6.7 inches, Full HD+ (1080 x 2340 pixels)', 'Tỷ lệ 19.5:9, mật độ điểm ảnh ~385 ppi', 'Corning Gorilla Glass Victus, Mohs level 5'], 'Độ phân giải': '1080 x 2340 pixels', 'Màn hình rộng': '6.7 inches', 'Công nghệ cảm ứng': 'Cảm ứng điện dung đa điểm'}}, {'category': 'Chụp hình & Quay phim', 'details

{'url': 'https://mobilecity.vn/dien-thoai/samsung-galaxy-a17-5g-chinh-hang.html',
 'name': 'Samsung Galaxy A17 5G Chính hãng',
 'price': '4.750.000 ₫',
 'specifications': [{'category': 'Thông tin chung',
   'details': {'Hệ điều hành': ['Android 15, One UI 7',
     'Hỗ trợ cập nhật 6 bản Android chính'],
    'Ngôn ngữ': 'Tiếng Việt, Đa ngôn ngữ'}},
  {'category': 'Màn hình',
   'details': {'Loại màn hình': 'Super AMOLED',
    'Màu màn hình': '16 triệu màu',
    'Chuẩn màn hình': ['Super AMOLED, 90Hz, 800 nits (HBM)',
     '6.7 inches, Full HD+ (1080 x 2340 pixels)',
     'Tỷ lệ 19.5:9, mật độ điểm ảnh ~385 ppi',
     'Corning Gorilla Glass Victus, Mohs level 5'],
    'Độ phân giải': '1080 x 2340 pixels',
    'Màn hình rộng': '6.7 inches',
    'Công nghệ cảm ứng': 'Cảm ứng điện dung đa điểm'}},
  {'category': 'Chụp hình & Quay phim',
   'details': {'Camera sau': ['50 MP, f/1.8 (góc rộng), 1/2.76", 0.64µm, AF, OIS',
     '5 MP, f/2.2 (góc siêu rộng), 1/5.0", 1.12µm',
     '2 MP, f/2.4, (m

Lấy ra 5 link sản phẩm từ file

In [ ]:
product_links = []
with open(mobilecity_links, "r") as f:
    reader = csv.reader(f)
    next(reader)  # Skip header
    for row in reader:
        product_links.append(row[0])
product_links[:5]

['https://mobilecity.vn/dien-thoai/apple-watch-series-5-40mm-vien-nhom-cu.html',
 'https://mobilecity.vn/dien-thoai/apple-watch-series-5-44mm-vien-nhom-cu.html',
 'https://mobilecity.vn/dien-thoai/samsung-galaxy-a17-5g-chinh-hang.html',
 'https://mobilecity.vn/dien-thoai/oneplus-ace-5-cu-99.html',
 'https://mobilecity.vn/dien-thoai/samsung-galaxy-s23-cu-man-amoled-120hz.html']

### Cuối cùng đây là hàm chính để chạy và cào dữ liệu, đảm bảo đã làm xong bước get Link và chạy các khai báo hàm rồi mới chạy được ở đây

-   Để giảm thời gian cào, ở đây nhóm sử dụng Đa luồng đa tuyến để cào dữ liệu. Lưu lại các Link lỗi để tiếp tục xử lí sau khi cào xong
-   Sử dụng 1 vài thư viện để log thông tin dễ theo dõi, debug trong quá trình cào dữ liệu


In [11]:
def crawl_products_multithreaded(
    product_links: list[str],
    max_workers: int = 5,
    batch_size: int = 10,
    delay_between_batches: int = 3,
) -> list[dict[str, Any]]:
    """
    Cào dữ liệu sản phẩm đa luồng từ danh sách các liên kết sản phẩm.

    Nếu một liên kết gặp lỗi, sẽ được thêm vào cuối danh sách để xử lý lại tuần tự sau.

    Args:
        product_links: Danh sách các URL sản phẩm cần cào
        max_workers: Số lượng luồng tối đa chạy đồng thời
        batch_size: Kích thước mỗi batch để tránh quá tải
        delay_between_batches: Thời gian chờ giữa các batch (giây)

    Returns:
        Danh sách các dictionary chứa thông tin sản phẩm
    """
    # Đảm bảo thư mục json tồn tại
    os.makedirs("../data/json", exist_ok=True)

    results = []
    retry_links = []  # Danh sách các liên kết cần thử lại

    # Chia danh sách sản phẩm thành các batch nhỏ hơn
    batches = [
        product_links[i : i + batch_size]
        for i in range(0, len(product_links), batch_size)
    ]

    print(f"Chia thành {len(batches)} batch, mỗi batch {batch_size} sản phẩm")

    def worker(link):
        try:
            # Khởi tạo driver riêng cho mỗi thread
            options = get_chrome_options()
            options.add_argument("--headless")

            driver = webdriver.Chrome(
                service=Service(ChromeDriverManager().install()), options=options
            )

            try:
                # Random Sleep trước mỗi request để tránh bị chặn
                time.sleep(random.uniform(1, 3))
                # Gọi hàm getAndSaveProductInfo để cào dữ liệu
                result = getAndSaveProductInfo(driver, link)
                return result
            finally:
                # Đảm bảo đóng driver sau khi sử dụng
                driver.quit()

        except Exception as e:
            error_msg = f"Lỗi khi xử lý link {link}: {str(e)}"
            print(error_msg)
            return {
                "url": link,
                "error": str(e),
                "specifications": {},
                "variant_prices": [],
            }

    # Xử lý theo từng batch với đa luồng
    for batch_idx, batch in enumerate(batches):
        print(f"Đang xử lý batch {batch_idx+1}/{len(batches)}")
        batch_results = []

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(worker, link): link for link in batch}
            for future in tqdm(
                as_completed(futures), total=len(batch), desc=f"Batch {batch_idx+1}"
            ):
                link = futures[future]
                try:
                    result = future.result()
                    batch_results.append(result)
                except Exception as e:
                    # Nếu có lỗi ngoài dự tính, báo lỗi và tiếp tục
                    print(f"Lỗi khi lấy kết quả cho {link}: {str(e)}")
                    batch_results.append(
                        {
                            "url": link,
                            "error": str(e),
                            "specifications": {},
                            "variant_prices": [],
                        }
                    )

        results.extend(batch_results)
        print(
            f"Đã cào {len(batch_results)}/{len(batch)} sản phẩm từ batch {batch_idx+1}"
        )
        print(f"Tổng số sản phẩm đã cào: {len(results)}/{len(product_links)}")

        if batch_idx < len(batches) - 1:
            print(
                f"Chờ {delay_between_batches} giây trước khi xử lý batch tiếp theo..."
            )
            time.sleep(delay_between_batches)

    # Xử lý các liên kết lỗi sau khi đa luồng bằng cách tuần tự
    # Nếu một liên kết lỗi, thêm nó vào cuối danh sách retry_links để thử lại sau
    retry_links = [result["url"] for result in results if result.get("error")]
    attempt = 1
    while retry_links:
        print(
            f"\nBắt đầu xử lý lại {len(retry_links)} liên kết lỗi (Lần thử: {attempt})"
        )
        current_retry = retry_links.copy()
        retry_links = []  # reset danh sách lỗi

        for link in tqdm(current_retry, desc=f"Retry attempt {attempt}"):
            try:
                options = get_chrome_options()
                options.add_argument("--headless")

                driver = webdriver.Chrome(
                    service=Service(ChromeDriverManager().install()), options=options
                )
                try:
                    # Cho phép load chậm hơn, không thêm độ trễ ngẫu nhiên
                    time.sleep(2)
                    result = getAndSaveProductInfo(driver, link)
                    # Cập nhật kết quả mới vào danh sách results:
                    # Tìm vị trí của kết quả cũ và thay thế
                    for idx, r in enumerate(results):
                        if r["url"] == link:
                            results[idx] = result
                            break
                finally:
                    driver.quit()
            except Exception as e:
                print(f"Lỗi khi xử lý lại link {link}: {str(e)}")
                # Nếu lỗi vẫn xảy ra, thêm link vào cuối danh sách để thử lại sau
                retry_links.append(link)
        if retry_links:
            print(
                f"Vẫn còn {len(retry_links)} liên kết lỗi, chờ 10 giây trước khi thử lại..."
            )
            time.sleep(10)
        attempt += 1

    # Lưu tất cả kết quả vào một file JSON
    try:
        os.makedirs("../data/processed", exist_ok=True)
        with open("../data/processed/mobilecity_data.json", "w", encoding="utf-8") as f:
            json.dump(results, f, indent=4, ensure_ascii=False)
        print(
            f"Đã lưu tất cả {len(results)} kết quả vào ../data/processed/mobilecity_data.json"
        )
    except Exception as e:
        print(f"Lỗi khi lưu ../data/processed/mobilecity_data.json: {str(e)}")

    return results

results = crawl_products_multithreaded(
    product_links=product_links[:10],
    max_workers=2,
    batch_size=5,
    delay_between_batches=10,
)

print(f"Hoàn thành cào dữ liệu cho {len(results)} sản phẩm")

Chia thành 2 batch, mỗi batch 5 sản phẩm
Đang xử lý batch 1/2


Batch 1:   0%|          | 0/5 [00:00<?, ?it/s]

Extracting data from: https://mobilecity.vn/dien-thoai/apple-watch-series-5-44mm-vien-nhom-cu.html
Found product name: Apple Watch Series 5 44mm Viền Nhôm cũ
Extracting data from: https://mobilecity.vn/dien-thoai/apple-watch-series-5-40mm-vien-nhom-cu.html
Found product name: Apple Watch Series 5 40mm Viền Nhôm cũ
Found product price: 4.150.000 ₫
Found product price: 3.950.000 ₫
product name common data: {'url': 'https://mobilecity.vn/dien-thoai/apple-watch-series-5-44mm-vien-nhom-cu.html', 'name': 'Apple Watch Series 5 44mm Viền Nhôm cũ', 'price': '4.150.000 ₫', 'specifications': [{'category': 'Thông tin chung', 'details': {'Hệ điều hành': 'WatchOS 6', 'Ngôn ngữ': 'Tiếng Anh, Tiếng Việt'}}, {'category': 'Màn hình', 'details': {'Loại màn hình': 'OLED, 16 triệu màu, 448 x 368 pixels, 1.78 inch', 'Màu màn hình': '16 triệu màu', 'Chuẩn màn hình': 'OLED, 16 triệu màu, 448 x 368 pixels, 1.78 inch', 'Độ phân giải': '448 x 368 pixels', 'Màn hình rộng': '1.78 inch', 'Công nghệ cảm ứng': 'Cảm ứ

Batch 1:  40%|████      | 2/5 [00:15<00:19,  6.40s/it]

Extracting data from: https://mobilecity.vn/dien-thoai/samsung-galaxy-a17-5g-chinh-hang.html
Found product name: Samsung Galaxy A17 5G Chính hãng
Extracting data from: https://mobilecity.vn/dien-thoai/oneplus-ace-5-cu-99.html
Found product name: OnePlus Ace 5 cũ (99% - Snapdragon 8 Gen 3)
Found product price: 4.750.000 ₫
Found product price: 6.650.000 ₫
product name common data: {'url': 'https://mobilecity.vn/dien-thoai/samsung-galaxy-a17-5g-chinh-hang.html', 'name': 'Samsung Galaxy A17 5G Chính hãng', 'price': '4.750.000 ₫', 'specifications': [{'category': 'Thông tin chung', 'details': {'Hệ điều hành': ['Android 15, One UI 7', 'Hỗ trợ cập nhật 6 bản Android chính'], 'Ngôn ngữ': 'Tiếng Việt, Đa ngôn ngữ'}}, {'category': 'Màn hình', 'details': {'Loại màn hình': 'Super AMOLED', 'Màu màn hình': '16 triệu màu', 'Chuẩn màn hình': ['Super AMOLED, 90Hz, 800 nits (HBM)', '6.7 inches, Full HD+ (1080 x 2340 pixels)', 'Tỷ lệ 19.5:9, mật độ điểm ảnh ~385 ppi', 'Corning Gorilla Glass Victus, Mohs l

...

Extracting data from: https://mobilecity.vn/dien-thoai/xiaomi-redmi-k80-cu.html
Found product name: Xiaomi REDMI K80 cũ (99% - Snapdragon 8 Gen 3)
Extracting data from: https://mobilecity.vn/dien-thoai/vivo-x100s-cu-gia-re.html
Found product name: Vivo X100s cũ (Đẹp 99% - Dimensity 9300 Plus)
Found product price: 6.950.000 ₫
Found product price: 11.450.000 ₫
product name common data: {'url': 'https://mobilecity.vn/dien-thoai/xiaomi-redmi-k80-cu.html', 'name': 'Xiaomi REDMI K80 cũ (99% - Snapdragon 8 Gen 3)', 'price': '6.950.000 ₫', 'specifications': [{'category': 'Thông tin chung', 'details': {'Hệ điều hành': 'Android 15, HyperOS 2.0', 'Ngôn ngữ': 'Đa ngôn ngữ'}}, {'category': 'Màn hình', 'details': {'Loại màn hình': 'OLED', 'Màu màn hình': '68 tỷ màu', 'Chuẩn màn hình': ['OLED, 68 tỷ màu, 120Hz, Dolby Vision, HDR10+, 1800 nits (HBM), 3200 nits (peak)', '6.67 inches, 2K (1440 x 3200 pixels)', 'Tỷ lệ 20:9, mật độ điểm ảnh ~526 ppi'], 'Độ phân giải': '1440 x 3200 pixels', 'Màn hình rộng'

Batch 2:  80%|████████  | 4/5 [00:27<00:05,  5.69s/it]

Extracting data from: https://mobilecity.vn/dien-thoai/zte-nubia-red-magic-9-pro-5g.html
Found product name: Nubia Red Magic 9 Pro 5G (Snapdragon 8 Gen 3)
Found product price: 13.000.000 ₫
product name common data: {'url': 'https://mobilecity.vn/dien-thoai/zte-nubia-red-magic-9-pro-5g.html', 'name': 'Nubia Red Magic 9 Pro 5G (Snapdragon 8 Gen 3)', 'price': '13.000.000 ₫', 'specifications': [{'category': 'Thông tin chung', 'details': {'Hệ điều hành': 'Android 14, Redmagic OS 9.0', 'Ngôn ngữ': 'Đa ngôn ngữ'}}, {'category': 'Màn hình', 'details': {'Loại màn hình': 'AMOLED', 'Màu màn hình': '1 tỷ màu', 'Chuẩn màn hình': ['AMOLED, 1 tỷ màu, 120Hz, 1600 nits (tối đa)', '6.8 inches, Full HD+ (1116 x 2480 pixels)', 'Tỷ lệ 20:9, mật độ điểm ảnh ~400 ppi'], 'Độ phân giải': '1116 x 2480 pixels', 'Màn hình rộng': '6.8 inches', 'Công nghệ cảm ứng': 'Cảm ứng điện dung đa điểm'}}, {'category': 'Chụp hình & Quay phim', 'details': {'Camera sau': ['50 MP (góc rộng), PDAF, OIS', '50 MP (góc siêu rộng)', 

Batch 2: 100%|██████████| 5/5 [00:39<00:00,  7.88s/it]

Đã cào 5/5 sản phẩm từ batch 2
Tổng số sản phẩm đã cào: 10/10
Đã lưu tất cả 10 kết quả vào ../data/processed/mobilecity_data.json
Hoàn thành cào dữ liệu cho 10 sản phẩm
